# Subclassification for Specific User Intent Categories

In [1]:
import pandas as pd

df_classifications = pd.read_csv(
    "../data/classifications/classifications_for_analysis.csv"
)

## Unit Functions for OpenAI Batch Classification

In [2]:
import asyncio
import json
from pathlib import Path

from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm

client = AsyncOpenAI()


async def classify_single_async(
    id_: int,
    message: str,
    system_prompt: str,
    schema: dict,
    model: str = "gpt-5-mini",
    max_retries: int = 3,
) -> tuple[int, dict | None, dict | None]:
    """
    Classify a single message asynchronously with retry.
    Returns (id, parsed_result, raw_response_dict).
    """
    for attempt in range(max_retries):
        try:
            response = await client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": message},
                ],
                response_format=schema,
            )
            parsed = json.loads(response.choices[0].message.content)
            raw = response.model_dump()
            return id_, parsed, raw
        except Exception as e:
            print(
                f"\n  [id={id_} attempt {attempt+1}/{max_retries}] {e} | {message[:60]!r}"
            )
            if attempt < max_retries - 1:
                await asyncio.sleep(2**attempt)  # exponential backoff: 1s, 2s

    return id_, None, None


async def classify_batch(
    messages: list[str],
    system_prompt: str,
    schema: dict,
    output_path: str,
    model: str = "gpt-5-mini",
    batch_size: int = 64,
    resume: bool = True,
) -> list[dict | None]:
    # Load already-completed ids
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    completed = {}
    if resume and output_path.exists():
        with open(output_path, "r", encoding="utf-8") as f:
            for line in f:
                rec = json.loads(line)
                completed[rec["id"]] = rec["result"]
        print(f"Resuming: {len(completed)} already completed, skipping.")

    results = [None] * len(messages)
    for id_, result in completed.items():
        if id_ < len(results):
            results[id_] = result

    total = len(messages)
    todo = [i for i in range(total) if i not in completed]

    if not todo:
        print("All messages already classified.")
        return results

    print(f"Classifying {len(todo)} messages in batches of {batch_size}...")

    with open(output_path, "a", encoding="utf-8") as f:
        for start in range(0, len(todo), batch_size):
            batch_ids = todo[start : start + batch_size]

            tasks = [
                classify_single_async(i, messages[i], system_prompt, schema, model)
                for i in batch_ids
            ]

            batch_results = await atqdm.gather(
                *tasks, desc=f"Batch {start//batch_size + 1}"
            )

            n_failed = 0
            for id_, parsed, raw in batch_results:
                results[id_] = parsed
                if parsed is None:
                    n_failed += 1
                record = {
                    "id": id_,
                    "message": messages[id_],
                    "result": parsed,
                    "raw": raw,
                }
                f.write(json.dumps(record, ensure_ascii=False) + "\n")
            f.flush()

            if n_failed:
                print(f"  ⚠ {n_failed} failed after all retries in this batch")

    return results

In [3]:
import json
from pathlib import Path


# Default pricing for gpt-5-mini (per 1M tokens)
PRICING = {
    "gpt-5-mini": {
        "input": 0.25,
        "cached_input": 0.025,
        "output": 2.00,
    },
}


def compute_cost(
    jsonl_path: str,
    model: str = "gpt-5-mini",
    batch_discount: bool = False,
) -> None:
    """
    Read a saved classification JSONL and compute token usage and cost.

    Args:
        jsonl_path:      path to .jsonl file saved by classify_batch
        model:           model name, used to look up pricing
        batch_discount:  if True, apply 50% batch API discount

    Returns:
        None (prints a cost report)
    """
    records = []
    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            if rec.get("raw"):
                records.append(rec["raw"])

    prompt_tokens = sum(r["usage"]["prompt_tokens"] for r in records)
    completion_tokens = sum(r["usage"]["completion_tokens"] for r in records)
    cached_tokens = sum(
        r["usage"].get("prompt_tokens_details", {}).get("cached_tokens", 0)
        for r in records
    )

    pricing = PRICING.get(model, PRICING["gpt-5-mini"])
    input_cost = (prompt_tokens - cached_tokens) * pricing["input"] / 1_000_000
    cached_cost = cached_tokens * pricing["cached_input"] / 1_000_000
    output_cost = completion_tokens * pricing["output"] / 1_000_000
    total_cost = input_cost + cached_cost + output_cost

    if batch_discount:
        total_cost *= 0.5

    n_total = len(records)
    n_failed = sum(
        1
        for line in open(jsonl_path, encoding="utf-8")
        if json.loads(line).get("result") is None
    )

    print(f"{'='*50}")
    print(f"Cost report: {Path(jsonl_path).name}")
    print(f"{'='*50}")
    print(f"Records:           {n_total} total, {n_failed} failed")
    print(f"Prompt tokens:     {prompt_tokens:,}")
    print(f"Completion tokens: {completion_tokens:,}")
    print(f"Cached tokens:     {cached_tokens:,}")
    print(f"---")
    print(f"Input cost:        ${input_cost:.4f}")
    print(f"Cached cost:       ${cached_cost:.4f}")
    print(f"Output cost:       ${output_cost:.4f}")
    if batch_discount:
        print(f"Batch discount:    50%")
    print(f"Total cost:        ${total_cost:.4f}")
    print(f"{'='*50}")

## Sentiment Expression Sub-classification

In [4]:
SENTIMENT_SYSTEM_PROMPT = """You are a sentiment classifier. Given a short user message, classify its sentiment.

Respond in JSON with exactly these fields:
{
  "reasoning": one sentence explanation of the sentiment classification,
  "label": "positive" | "negative" | "neutral",
  "custom_label": a single word describing the specific emotion (e.g. anger, frustration, satisfaction, gratitude, confusion, excitement, greeting, ...)
}
"""

SENTIMENT_SCHEMA = {
    "type": "json_schema",
    "json_schema": {
        "name": "sentiment_classification",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "label": {
                    "type": "string",
                    "enum": ["positive", "negative", "neutral"],
                },
                "custom_label": {"type": "string"},
                "reasoning": {"type": "string"},
            },
            "required": ["label", "custom_label", "reasoning"],
            "additionalProperties": False,
        },
    },
}

In [5]:
import os

os.makedirs("../data/sub_classifications/sentiment_expression", exist_ok=True)
sentiment_jsonl_path = "../data/sub_classifications/sentiment_expression/raw.jsonl"

sentiment_messages = df_classifications[
    df_classifications["sub_category"] == "7.5 Sentiment Expression"
]["truncated_content"].tolist()

results = await classify_batch(
    sentiment_messages,
    SENTIMENT_SYSTEM_PROMPT,
    SENTIMENT_SCHEMA,
    output_path=sentiment_jsonl_path,
    model="gpt-5-mini",
)

compute_cost(sentiment_jsonl_path, model="gpt-5-mini", batch_discount=False)

Resuming: 1535 already completed, skipping.
All messages already classified.
Cost report: raw.jsonl
Records:           1535 total, 0 failed
Prompt tokens:     329,807
Completion tokens: 415,469
Cached tokens:     0
---
Input cost:        $0.0825
Cached cost:       $0.0000
Output cost:       $0.8309
Total cost:        $0.9134
